In [3]:
import pandas as pd
import ast
import numpy as np
from fuzzywuzzy import fuzz  # Fuzzy matching için

#########################
# 1) RAW CSV'Yİ OKU (Kirli CSV)
#########################
# Bu CSV'de sütunlar: [name, place_id, rating, total_ratings, address, phone, reviews]
# 'reviews' sütununda JSON formatında yorumlar var.
raw_csv_path = "C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\mugla_hotels_expanded2 - mugla_hotels_expanded2.csv"
df_raw = pd.read_csv(raw_csv_path, encoding="utf-8-sig")

raw_reviews_list = []
for idx, row in df_raw.iterrows():
    hotel_name = row["name"]                     # Otelin adı
    hotel_overall_rating = row["rating"]         # Otelin genel rating puanı
    reviews_json_str = row["reviews"]            # JSON formatındaki yorumlar
    
    if pd.isna(reviews_json_str):
        continue
    try:
        reviews_data = ast.literal_eval(reviews_json_str)
    except Exception as e:
        continue
    
    for rev in reviews_data:
        review_author = rev.get("author_name", "")
        review_rating = rev.get("rating", None)
        review_text   = rev.get("text", "").strip()
        review_lang   = rev.get("language", "")
        
        raw_reviews_list.append({
            "hotel_name": hotel_name,
            "hotel_overall_rating": hotel_overall_rating,
            "review_author_name": review_author,
            "review_rating": review_rating,
            "review_text": review_text,
            "review_language": review_lang
        })

raw_reviews_df = pd.DataFrame(raw_reviews_list)

#########################
# 2) FINAL CSV'Yİ OKU (Temizlenmiş CSV)
#########################
# Bu CSV'de sütunlar: [hotel_name, author_name, language, review_text, processed_review]
# Burada 'review_text' sütunu orijinaline daha yakın; processed_review ise çok temizlenmiş.
final_csv_path = "C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_deepl_use.csv"
final_df = pd.read_csv(final_csv_path, encoding="utf-8-sig")

# Yeni sütunları ekleyelim
final_df["hotel_overall_rating"] = np.nan
final_df["review_rating"] = np.nan

#########################
# 3) EŞLEŞTİRME FONKSİYONU (Fuzzy Matching ile)
#########################
def find_ratings_for_row(row, raw_df):
    hotel_name = row["hotel_name"]
    author_name = str(row["author_name"]).strip()
    lang = str(row["language"]).strip()
    final_text = str(row["review_text"]).strip().lower()  # Daha az temizlenmiş metin kullanıyoruz
    
    # Otel adı aynı olanları filtrele
    subset = raw_df[raw_df["hotel_name"] == hotel_name]
    
    # Yazar adı ve dil eşleşmesini uygula (varsa)
    if author_name:
        subset = subset[subset["review_author_name"] == author_name]
    if lang:
        subset = subset[subset["review_language"] == lang]
    
    if subset.empty:
        return pd.Series([np.nan, np.nan])
    
    best_score = 0
    best_match = None
    # Fuzzy matching ile benzerlik skorunu hesapla
    for i, r in subset.iterrows():
        raw_text = r["review_text"].lower().strip()
        score = fuzz.ratio(final_text, raw_text)
        if score > best_score:
            best_score = score
            best_match = r
    
    # Eşik değeri (örn. 70) üzerinde ise eşleşme kabul edilsin
    threshold = 70
    if best_score >= threshold and best_match is not None:
        return pd.Series([best_match["hotel_overall_rating"], best_match["review_rating"]])
    else:
        return pd.Series([np.nan, np.nan])

#########################
# 4) FINAL DF ÜZERİNDE UYGULA
#########################
final_df[["hotel_overall_rating", "review_rating"]] = final_df.apply(lambda row: find_ratings_for_row(row, raw_reviews_df), axis=1)

#########################
# 5) YENİ CSV'Yİ KAYDET
#########################
updated_csv_path = "C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_with_ratings.csv"
final_df.to_csv(updated_csv_path, index=False, encoding="utf-8-sig")
print("Updated CSV saved at:", updated_csv_path)


C:\Users\catsu\AppData\Local\Programs\Python\Python311\Lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


Updated CSV saved at: C:\Users\catsu\PycharmProjects\scrappyhotel\final_processed_reviews_with_ratings.csv


In [4]:
import pandas as pd
import ast
import numpy as np
from fuzzywuzzy import fuzz

##############################################
# 1) RAW CSV'DEN RATING BİLGİLERİNİ ÇEK
##############################################
# Raw CSV sütun yapısı:
# [name, place_id, rating, total_ratings, address, phone, reviews]
# reviews sütununda JSON formatında yorumlar var.
raw_csv_path = "C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\mugla_hotels_expanded2 - mugla_hotels_expanded2.csv"
df_raw = pd.read_csv(raw_csv_path, encoding="utf-8-sig")

# Her otel için JSON içindeki tüm yorumları satır bazında çıkartıyoruz.
raw_reviews_list = []

for idx, row in df_raw.iterrows():
    hotel_name = str(row["name"]).strip()                # Otel adı
    hotel_overall_rating = row["rating"]                 # Otelin genel puanı
    reviews_json_str = row["reviews"]                    # JSON formatındaki yorumlar
    
    if pd.isna(reviews_json_str):
        continue
    try:
        reviews_data = ast.literal_eval(reviews_json_str)
    except Exception as e:
        continue  # JSON parse hatası varsa atla
    
    # Her yorumu ayrı satır olarak ekle
    for rev in reviews_data:
        review_author = rev.get("author_name", "").strip()
        review_rating = rev.get("rating", None)
        review_text   = rev.get("text", "").strip()
        review_lang   = rev.get("language", "").strip()
        
        raw_reviews_list.append({
            "hotel_name": hotel_name,
            "hotel_overall_rating": hotel_overall_rating,
            "review_author_name": review_author,
            "review_rating": review_rating,
            "review_text": review_text,
            "review_language": review_lang
        })

raw_reviews_df = pd.DataFrame(raw_reviews_list)
print("Raw reviews dataframe shape:", raw_reviews_df.shape)

##############################################
# 2) FINAL CSV'DEN TEMİZLENMİŞ VERİYİ OKU
##############################################
# Final CSV sütunları örneğin: 
# [hotel_name, author_name, language, review_text, processed_review]
final_csv_path = "C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_deepl_use.csv"
final_df = pd.read_csv(final_csv_path, encoding="utf-8-sig")

# Yeni rating sütunlarını ekleyelim
final_df["hotel_overall_rating"] = np.nan
final_df["review_rating"] = np.nan

##############################################
# 3) Fuzzy Matching İLE EŞLEŞTİRME FONKSİYONU
##############################################
def find_ratings_for_row(row, raw_df, threshold=70):
    """
    Final CSV'deki bir satırın review_text'i ile, aynı otel için raw_df'deki yorumlar arasında
    fuzzy matching yaparak (fuzz.ratio ve token_set_ratio kullanarak) en yüksek skoru alan eşleşmeyi bulur.
    Eğer skor belirlenen eşik (threshold) üzerinde ise,
    otelin genel rating ve yorum rating'ini döndürür.
    """
    hotel_name = str(row["hotel_name"]).strip()
    author_name = str(row["author_name"]).strip()
    lang = str(row["language"]).strip()
    # Daha az temizlenmiş review_text'i kullanıyoruz
    final_text = str(row["review_text"]).strip().lower()
    
    # Öncelikle otel adına göre filtrele
    subset = raw_df[raw_df["hotel_name"].str.strip() == hotel_name]
    
    # Eğer varsa, yazar adı ve dil ile daralt
    if author_name:
        subset = subset[subset["review_author_name"].str.strip() == author_name]
    if lang:
        subset = subset[subset["review_language"].str.strip() == lang]
    
    if subset.empty:
        return pd.Series([np.nan, np.nan])
    
    best_score = 0
    best_match = None
    # Her bir raw yorumla fuzzy matching yap
    for i, r in subset.iterrows():
        raw_text = str(r["review_text"]).strip().lower()
        # Farklı fuzzy metrikleri kullan: ratio ve token_set_ratio
        score1 = fuzz.ratio(final_text, raw_text)
        score2 = fuzz.token_set_ratio(final_text, raw_text)
        score = (score1 + score2) / 2  # Ortalama skor
        if score > best_score:
            best_score = score
            best_match = r
    
    # Eğer en iyi skor eşik değerin üzerinde ise, rating bilgilerini döndür
    if best_score >= threshold and best_match is not None:
        return pd.Series([best_match["hotel_overall_rating"], best_match["review_rating"]])
    else:
        return pd.Series([np.nan, np.nan])

##############################################
# 4) FINAL DF ÜZERİNDE UYGULA
##############################################
final_df[["hotel_overall_rating", "review_rating"]] = final_df.apply(
    lambda row: find_ratings_for_row(row, raw_reviews_df, threshold=70), axis=1
)

##############################################
# 5) GÜNCELLENMİŞ CSV'Yİ KAYDET
##############################################
updated_csv_path = "C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_with_ratings2.csv"
final_df.to_csv(updated_csv_path, index=False, encoding="utf-8-sig")
print("Updated CSV saved at:", updated_csv_path)


Raw reviews dataframe shape: (1956, 6)
Updated CSV saved at: C:\Users\catsu\PycharmProjects\scrappyhotel\final_processed_reviews_with_ratings2.csv
